In [ ]:
import random
import time

In [ ]:
# for implementing sliding, tumbling windows, generic buffers etc.
class Buffer:
    def __init__(self):
        self.data = []

    #insert list of values at end
    def insert(self, values):
        self.data.extend(values)

    #remove count values from beginning
    def remove(self, count):
        if count <= len(self.data):
            del self.data[:count]

    def size(self):
        return len(self.data)

    def clear(self):
        self.data = []

# stream of numbers - uniform distribution
class UniformNumericStream:
    def __init__(self, minVal, maxVal, speed = 1):
        self.minVal = minVal
        self.maxVal = maxVal
        self.speed = speed

    def next(self):
        if self.speed <= 0:
            delay = 0
        else:
            delay = random.random() / self.speed
            time.sleep(delay)
        newVal = self.minVal + random.random() * (self.maxVal - self.minVal)
        return newVal, delay

# stream of chars from text file
class CharStream:
    def __init__(self, fileName, speed = 1):
        f = open(fileName, 'r')
        self.speed = speed
        self.textData = f.readlines()
        self.textData = self.textData[0] #first and only line
        self.totalCharCount = len(self.textData)
        self.charCounter = 0

    def next(self):
        if self.speed <= 0:
            delay = 0
        else:
            delay = random.random() / self.speed
            time.sleep(delay)
        newChar = self.textData[self.charCounter]
        self.charCounter = self.charCounter+1
        return newChar, delay


# Task 1
 Given a stream of real values, calculate the average value of the streamed data and display it at certain
time intervals of your choice. Since the data stream is theoretically infinite, do not compute or update the
sum of the values for every new data element (you may count the values, however). The average has to be
updated on each new value. Implement the computations using both an „element by element” and a
tumbling window approach. For a stream of uniformly-distributed numbers, after a while it is expected
that the average will stay approximately the same regardless of the number of values.

In [ ]:
# „element by element” approach
stream = UniformNumericStream(minVal=0, maxVal=10, speed=0)

avg   = 0.0   # start with average = 0
count = 0     # no values seen yet

print("Part A – element by element")
print(f"{'Count':>6}  |  {'Average':>10} \n")

for i in range(200):

    new_value = stream.next()[0]   # get next number from the stream

    count += 1
    avg = avg + (new_value - avg) / count

    if count % 20 == 0:
        print(f"{count:>6}  |  {avg:>10.4f}")

print(f"\nAfter {count} values, average = {avg:.4f}")

Part A – element by element
 Count  |     Average 

    20  |      5.3031
    40  |      4.8457
    60  |      4.6701
    80  |      4.4804
   100  |      4.7036
   120  |      4.8601
   140  |      4.8713
   160  |      4.7587
   180  |      4.6947
   200  |      4.6691

After 200 values, average = 4.6691


In [ ]:
# tumbling window approach.

stream = UniformNumericStream(minVal=0, maxVal=10, speed=0)
buffer = Buffer()

window_size = 20
window_num  = 0

print("Part B – tumbling window (size = 20)")
print(f"{'Window':>8}  |  {'Average':>10}")

for i in range(200):

    new_value    = stream.next()[0]

    buffer.insert([new_value])

    if buffer.size() >= window_size:
        window_num += 1
        total = 0
        for v in buffer.data:
            total += v
        window_avg = total / buffer.size()

        print(f"{window_num:>8}  |  {window_avg:>10.4f}")
        buffer.clear()

Part B – tumbling window (size = 20)
  Window  |     Average
       1  |      4.8851
       2  |      4.9501
       3  |      5.9220
       4  |      4.9773
       5  |      4.4647
       6  |      5.8280
       7  |      4.4338
       8  |      5.2072
       9  |      4.2661
      10  |      5.1026


# Task 2
Given a stream of bits, calculate and display the 32 bit unsigned integers that result from the bit stream
(i.e. combine every 32 bits from the stream into a 32 bit integer). Assume Big Endian byte order (the first
8 bits are the most significant byte, followed by the others in descending order of their significance)

In [ ]:
stream = UniformNumericStream(minVal=0, maxVal=1, speed=0) # will use to generate bits
buffer = Buffer()

integer_num = 0
n_integers_to_produce = 10

print(f"{'Integer #':>10}  |  {'Value':>12}  |  Bits\n")


while integer_num < n_integers_to_produce:

    value = stream.next()[0]
# the stream will have float values between 0 and 1, so we can "convert" them in bits
    if value < 0.5:
        bit = 0
    else:
        bit = 1
    buffer.insert([bit])

    if buffer.size() >= 32:
        integer_num += 1

        result = 0
        for i in range(32):
            power  = 31 - i
            result = result + buffer.data[i] * (2 ** power)
        bits_string = ""
        for b in buffer.data:
            bits_string += str(b)

        print(f"{integer_num:>10}  |  {result:>12}  |  {bits_string}")

        buffer.clear()


 Integer #  |         Value  |  Bits

         1  |       4051033  |  00000000001111011101000001011001
         2  |    3457577949  |  11001110000101100111001111011101
         3  |    1472033277  |  01010111101111010111000111111101
         4  |    2296834412  |  10001000111001101110100101101100
         5  |    1695781969  |  01100101000100111001010001010001
         6  |      54161065  |  00000011001110100110111010101001
         7  |     754497639  |  00101100111110001011100001100111
         8  |    3947658756  |  11101011010011000111111000000100
         9  |     806177583  |  00110000000011010100101100101111
        10  |     619313089  |  00100100111010011111011111000001
Done! Produced 10 integers.


# Task 4
 Given a stream of characters representing a text, calculate the counts of each distinct word determined
from the stream.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sampletext.txt to sampletext.txt


In [ ]:


stream = CharStream('sampletext.txt', speed=0)

word_counts  = {}    # word → number of apparences
current_word = ""
words_so_far = 0



for i in range(stream.totalCharCount):

    char = stream.next()[0]

    if char != ' ' and char != '\n' and char != ',' and char != '.':
        current_word += char

    else:
        if current_word != "":   # ignore multiple spaces

            current_word = current_word.lower() # "Begin" and "begin" are the same word

            if current_word not in word_counts:
                word_counts[current_word] = 0
            word_counts[current_word] += 1

            words_so_far += 1
            current_word = ""

            if words_so_far % 20 == 0:
                print(f"After {words_so_far} words:")
                for word, count in word_counts.items():
                    print(f"  '{word}' → {count}\n")


print(f"Final word counts after {words_so_far} words:")
for word, count in word_counts.items():
    print(f"  '{word}' → {count}")

After 20 words:
  'computer' → 3

  'software' → 2

  'or' → 2

  'simply' → 1

  'is' → 1

  'a' → 1

  'collection' → 1

  'of' → 1

  'data' → 1

  'instructions' → 1

  'that' → 1

  'tell' → 1

  'the' → 1

  'how' → 1

  'to' → 1

  'work' → 1

After 40 words:
  'computer' → 4

  'software' → 2

  'or' → 2

  'simply' → 1

  'is' → 3

  'a' → 1

  'collection' → 1

  'of' → 1

  'data' → 1

  'instructions' → 1

  'that' → 1

  'tell' → 1

  'the' → 3

  'how' → 1

  'to' → 2

  'work' → 2

  'this' → 1

  'in' → 2

  'contrast' → 1

  'physical' → 1

  'hardware' → 1

  'from' → 1

  'which' → 1

  'system' → 1

  'built' → 1

  'and' → 1

  'actually' → 1

  'performs' → 1

After 60 words:
  'computer' → 8

  'software' → 5

  'or' → 2

  'simply' → 1

  'is' → 4

  'a' → 1

  'collection' → 1

  'of' → 1

  'data' → 2

  'instructions' → 1

  'that' → 1

  'tell' → 1

  'the' → 3

  'how' → 1

  'to' → 2

  'work' → 2

  'this' → 1

  'in' → 2

  'contrast' → 1

  'physical' →